In [ ]:
# @title 0. Montage Google Drive (Si exécution sur Colab)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive monté avec succès !")
except ImportError:
    print("Environnement hors Colab détecté. Montage Drive ignoré.")


# HGP-clusterer : 3D puis 4D Panoptic Segmentation sur SemanticKITTI

Ce notebook implémente un pipeline de segmentation panoptique en "streaming" en utilisant **HGP-clusterer**.

**Pipeline :**
1.  **Setup** : Installation des dépendances.
2.  **Data** : Chargement d'une séquence SemanticKITTI.
3.  **Preprocessing** : Construction des nuages de points.
4.  **Clustering & Tracking (Streaming)** : 
    - Initialisation (Frame 0) : Segmentation d'instance 3D (ou BEV) et création des représentations des clusters (volume, classe, vitesse).
    - Suivi (Frames suivantes) : *En cours (itération frame par frame)*.
5.  **Evaluation** : *Adaptée pour le suivi*.
6.  **Visualisation** : Rendu interactif.

In [ ]:
# @title 1.1 Choix du Backend Géométrique
# 'geogram' est recommandé pour la vitesse (headless). 'cgal' est plus lent mais exact.
BACKEND = 'cgal'  # @param ['geogram', 'cgal']
print(f"Backend sélectionné : {BACKEND}")

In [ ]:
# @title 1.2 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev

if BACKEND == 'cgal':
    # libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
    !apt-get install -y -qq libcgal-dev libtbb-dev libtbbmalloc2 libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.3 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml shapely

In [ ]:
%%bash
# @title 1.4 Installation de HGP-clusterer et SemanticKITTI-API
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi

# SemanticKITTI API (pour l'évaluation)
if [ -d semantic-kitti-api ]; then
    git -C semantic-kitti-api pull --ff-only
else
    git clone https://github.com/PRBonn/semantic-kitti-api.git
fi

In [ ]:
# @title 1.5 Compilation de HGP
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

if BACKEND == 'geogram':
    if not os.path.exists('geogram'):
        print("Clonage de Geogram...")
        !git clone --recursive https://github.com/BrunoLevy/geogram.git
    
    print("Compilation de Geogram (Headless)...")
    !cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_GARGANTUA=OFF
    !cmake --build geogram/build --config Release --parallel 4
    !cmake --install geogram/build --prefix /usr/local
    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'

elif BACKEND == 'cgal':
    print("Configuration CGAL...")
    
    # -- FIX: Add CGAL path to environment for setup_cgal.py --
    cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
    current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
    os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
    # ---------------------------------------------------------

    # On tente de construire l'outil CGAL, mais on continue même en cas d'erreur
    # car le setup.py principal pourrait réussir autrement.
    try:
        subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
    except subprocess.CalledProcessError:
        print("⚠️ Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
if BACKEND == 'geogram':
    install_cmd = f"GEOGRAM_INSTALL_PREFIX=/usr/local {install_cmd}"
elif BACKEND == 'cgal':
    # Ajout du chemin système pour CGAL (Debian/Ubuntu/Colab)
    # Note: On le passe aussi explicitement ici pour être sûr
    install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_clusterer import HGPClusterer
    print("✅ HGPClusterer installé.")
except ImportError as e:
    print(f"❌ Erreur import HGP: {e}")

In [ ]:
# @title 2.1 Configuration Séquence et Téléchargement
# IMPORTANT : Si vous ne voulez tester qu'une seule séquence, lancez cette cellule.
# Le téléchargement via gdown --folder récupère tout le dossier si on ne filtre pas.
# Ici, on télécharge tout le dataset SemanticKITTI (partiel) fourni via le lien Drive.

SEQUENCE_TO_TEST = 8 # @param {type:"integer"}
DOWNLOAD_DATA = True # @param {type:"boolean"}

START_FRAME = 0 # @param {type:"integer"}
NUM_FRAMES = 200 # @param {type:"integer"} (-1 pour toutes les frames)
DT_SCALE = 0.1  # @param {type:"number"}
APPLY_BEV = False # @param {type:"boolean"}
# Mode Sémantique :
# - 'Oracle' : Utilise la vérité terrain fournie par SemanticKITTI pour filtrer les objets mobiles (Things).
# - 'None' : Ne filtre rien (Lance le clustering sur absolument toute la scène, lent et non recommandé).
# (Note: Le chargement d'une prédiction réseau externe viendrait ici dans une future mise à jour)
SEMANTIC_MODE = "Oracle" # @param ["Oracle", "None"]

# Choix du mode de téléchargement :
# - 'Folder' : Télécharge fichier par fichier (Très lent pour 10k fichiers, mais utile si on a que le lien du dossier)
# - 'Zip' : Télécharge une archive unique et décompresse (Beaucoup plus rapide, recommandé)
DOWNLOAD_MODE = "Zip" # @param ["Folder", "Zip"]

# IDs Google Drive par séquence
# Remplissez ce dictionnaire avec les IDs des dossiers ou des zips pour chaque séquence.
SEQUENCE_DRIVE_IDS = {
    8: {
        "Folder": "1UqFKvekjyic6L_8KD1kcv8MuGmQMIk0A",
        "Zip": "1ZoZtzdFAkPWYHT8sFsHjwyEmFHbpaIQH"
    }
}

# Dossier Racine (Fallback si ID spécifique non trouvé en mode Folder)
ROOT_FOLDER_ID = "1ORVzSo-TWbNHeAC0-k3mxX9AiJHI_tVu"

if DOWNLOAD_DATA:
    import os
    import shutil
    
    # Destination racine
    base_dest = "/content/semantic_kitti_data"
    seq_str = f"{SEQUENCE_TO_TEST:02d}"
    target_dir = os.path.join(base_dest, seq_str)
    
    if not os.path.exists(target_dir):
        print(f"Démarrage du téléchargement (Mode : {DOWNLOAD_MODE})...")
        
        # Récupération des IDs pour la séquence choisie
        seq_ids = SEQUENCE_DRIVE_IDS.get(SEQUENCE_TO_TEST, {})
        
        if DOWNLOAD_MODE == "Zip":
            zip_id = seq_ids.get("Zip")
            if not zip_id:
                print(f"⚠️ Aucun ID Zip trouvé pour la séquence {SEQUENCE_TO_TEST}. Veuillez remplir SEQUENCE_DRIVE_IDS.")
                print("Passage automatique en mode Folder (Fallback)...")
                # On ne lance pas d'erreur, on essaie le folder si possible, sinon root
                DOWNLOAD_MODE = "Folder" 
            else:
                print(f"Téléchargement de l'archive Zip (ID: {zip_id})...")
                zip_path = os.path.join(base_dest, "sequence.zip")
                os.makedirs(base_dest, exist_ok=True)
                
                # Téléchargement via Gdown
                res = os.system(f"gdown {zip_id} -O {zip_path} --quiet")
                
                # FALLBACK GOOGLE DRIVE : Si gdown a échoué (erreur 256, quota dépassé, etc.)
                if res != 0 or not os.path.exists(zip_path):
                    print("\nErreur Gdown (Quota dépassé ou accès refusé).")
                    print("Tentative de montage de Google Drive pour récupérer l'archive locale...")
                    try:
                        from google.colab import drive
                        drive.mount('/content/drive')
                        local_zip = f"/content/drive/MyDrive/Datasets/semantic_kitti/sequences_zip/{seq_str}.zip"
                        if os.path.exists(local_zip):
                            print(f"Fichier trouvé sur Drive ({local_zip}), copie en cours...")
                            shutil.copy(local_zip, zip_path)
                        else:
                            print(f"Fichier introuvable sur Drive à l'emplacement : {local_zip}")
                            print("Le pipeline va probablement échouer.")
                    except ImportError:
                        print("Impossible de monter Google Drive (environnement hors Colab). Le pipeline va probablement échouer.")
                
                if os.path.exists(zip_path):
                    print("Décompression...")
                    # On décompresse
                    os.system(f"unzip -q {zip_path} -d {target_dir}")
                    os.system(f"rm -f {zip_path}")
                    
                    # Vérification de la structure (si le zip contenait un sous-dossier, on remonte)
                    if os.path.exists(target_dir) and not os.path.exists(os.path.join(target_dir, "velodyne")):
                        # Tentative de correction automatique
                        sub_dirs = [d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))]
                        if len(sub_dirs) >= 1:
                            inner_dir = os.path.join(target_dir, sub_dirs[0])
                            print(f"Structure imbriquée détectée, déplacement de {inner_dir} vers {target_dir}...")
                            for item in os.listdir(inner_dir):
                                shutil.move(os.path.join(inner_dir, item), target_dir)
                            os.rmdir(inner_dir)

        # Note: Ce bloc est exécuté si mode Folder OU si fallback depuis Zip
        if DOWNLOAD_MODE == "Folder":
            folder_id = seq_ids.get("Folder")
            if not folder_id:
                # Fallback sur le root folder (pas idéal mais fonctionnel)
                print(f"ID spécifique Folder manquant pour la séquence {SEQUENCE_TO_TEST}.")
                folder_id = ROOT_FOLDER_ID
                dl_target = base_dest
            else:
                dl_target = target_dir

            print(f"Téléchargement du dossier (ID: {folder_id})...")
            # --- TÉLÉCHARGEMENT SÉLECTIF ---
            try:
                import gdown
                print("Analyse du contenu du Drive... (limité aux 50 premiers fichiers par dossier par Google Drive)")
                # remaining_ok=True permet de récupérer les 50 premiers fichiers sans crasher
                files = gdown.download_folder(id=folder_id, skip_download=True, quiet=True, remaining_ok=True)
                if files:
                    needed_frames = set()
                    if NUM_FRAMES != -1:
                        needed_frames = set(range(START_FRAME, START_FRAME + NUM_FRAMES))
                    
                    downloaded_count = 0
                    
                    for f in files:
                        try:
                            f_path = f.path if hasattr(f, 'path') else f.get('path', '')
                            f_id = f.id if hasattr(f, 'id') else f.get('id', '')
                        except:
                            continue
                            
                        # Vérifier si c'est un fichier lié à une frame
                        filename = f_path.split('/')[-1] if '/' in f_path else f_path
                        is_frame_file = False
                        frame_idx = -1
                        
                        if filename.endswith('.bin') or filename.endswith('.label'):
                            try:
                                frame_idx = int(filename.split('.')[0])
                                is_frame_file = True
                            except ValueError:
                                pass
                                
                        # Filtrer
                        if is_frame_file and NUM_FRAMES != -1:
                            if frame_idx not in needed_frames:
                                continue # On skip cette frame
                        
                        # Créer le chemin local
                        import os
                        local_path = os.path.join(dl_target, f_path)
                        os.makedirs(os.path.dirname(local_path), exist_ok=True)
                        
                        if not os.path.exists(local_path):
                            print(f"Téléchargement : {f_path}")
                            gdown.download(id=f_id, output=local_path, quiet=True)
                        downloaded_count += 1
                        
                    print(f"Téléchargement sélectif terminé ({downloaded_count} fichiers traités).")
                    
                    # Vérification si on a pu tout récupérer ou si on a tapé la limite des 50
                    # On s'attend à 2 fichiers par frame (bin + label) s'il s'agit de frames
                    # Mais s'il y a des fichiers autres (calib), on ne s'inquiète que si downloaded_count est faible.
                    # Pour être simple, on informe l'utilisateur si la frame demandée max n'est pas dans les 50.
                    if NUM_FRAMES != -1 and (START_FRAME + NUM_FRAMES) > 50:
                        print(f"\n⚠️ ATTENTION : Vous avez demandé jusqu'à la frame {START_FRAME + NUM_FRAMES - 1}.")
                        print("Google Drive limite le listage anonyme aux 50 premiers fichiers de chaque dossier (jusqu'à la frame 49).")
                        print("👉 Pour traiter au-delà de la frame 49, VEUILLEZ UTILISER LE MODE 'Zip' dans les paramètres.\n")
                else:
                    print("Impossible d'analyser le dossier. Veuillez utiliser le mode Zip.")
            except Exception as e:
                print(f"\n❌ Le téléchargement sélectif a échoué ({e}).")
                print("👉 VEUILLEZ UTILISER LE MODE 'Zip' DANS LES PARAMÈTRES CI-DESSUS.\n")
        print("Téléchargement terminé.")
    else:
        print(f"Dossier {target_dir} existe déjà. Skip download.")
else:
    print("Téléchargement désactivé.")

print(f"Séquence cible pour le test : {SEQUENCE_TO_TEST}")

In [ ]:
# @title 2.2 Loader SemanticKITTI
import os
import numpy as np
import glob

class SemanticKITTILoader:
    def __init__(self, base_path, sequence_num):
        self.seq_str = f"{sequence_num:02d}"

        # Recherche du dossier de la séquence.
        # Structure attendue : base_path/08 ou base_path/sequences/08

        # 1. Chercher direct
        possible_paths = glob.glob(f"{base_path}/{self.seq_str}")

        # 2. Chercher dans un sous-dossier 'sequences' (structure officielle KITTI)
        if not possible_paths:
            possible_paths = glob.glob(f"{base_path}/**/sequences/{self.seq_str}", recursive=True)

        # 3. Chercher récursivement n'importe où (au cas où gdown a créé une structure intermédiaire)
        if not possible_paths:
             possible_paths = glob.glob(f"{base_path}/**/{self.seq_str}", recursive=True)

        # Filtrer pour ne garder que les vrais dossiers contenant 'velodyne'
        valid_paths = []
        for p in possible_paths:
            if os.path.exists(os.path.join(p, 'velodyne')):
                valid_paths.append(p)

        if not valid_paths:
            raise ValueError(f"Séquence {self.seq_str} introuvable dans {base_path}. Vérifiez que le dossier 'velodyne' est bien présent.")

        self.seq_path = valid_paths[0]
        print(f"Séquence chargée : {self.seq_path}")

        self.velo_path = os.path.join(self.seq_path, 'velodyne')
        self.label_path = os.path.join(self.seq_path, 'labels')
        self.poses_file = os.path.join(self.seq_path, 'poses.txt')
        self.calib_file = os.path.join(self.seq_path, 'calib.txt')

        # Fallback pour poses.txt/calib.txt s'ils sont dans le dossier parent (structure dataset/sequences/08)
        if not os.path.exists(self.poses_file):
             # Essayer de remonter d'un niveau (dataset/sequences/) ou deux
             parent = os.path.dirname(self.seq_path) # dataset/sequences
             grandparent = os.path.dirname(parent) # dataset

             # Cas dataset/poses.txt (peu probable mais...)
             # Cas dataset/sequences/08/poses.txt (standard)
             pass

        self.scan_files = sorted(glob.glob(os.path.join(self.velo_path, '*.bin')))
        self.label_files = sorted(glob.glob(os.path.join(self.label_path, '*.label')))
        self.poses = self._load_poses()
        self.calib = self._load_calib()

    def _load_poses(self):
        if not os.path.exists(self.poses_file):
            print(f"Info: poses.txt non trouvé ({self.poses_file}).")
            return []
        poses = []
        with open(self.poses_file, 'r') as f:
            for line in f:
                values = [float(v) for v in line.strip().split()]
                pose = np.vstack([np.array(values).reshape(3, 4), [0, 0, 0, 1]])
                poses.append(pose)
        return poses

    def _load_calib(self):
        if not os.path.exists(self.calib_file):
            print(f"Info: calib.txt non trouvé ({self.calib_file}).")
            return np.eye(4)
        calib = {}
        with open(self.calib_file, 'r') as f:
            for line in f:
                if ':' not in line: continue
                key, val = line.split(':', 1)
                calib[key] = np.array([float(x) for x in val.split()]).reshape(3, 4)
        if 'Tr' in calib:
            return np.vstack([calib['Tr'], [0, 0, 0, 1]])
        return np.eye(4)

    def get_scan(self, idx, apply_pose=True):
        scan = np.fromfile(self.scan_files[idx], dtype=np.float32).reshape(-1, 4)
        points = scan[:, :3]
        if apply_pose and self.poses and idx < len(self.poses):
            # Transformation officielle SemanticKITTI : inv(Tr) @ Pose_Cam @ Tr
            # Cela permet de transformer les points Lidar vers le monde tout en restant 
            # dans le système de coordonnées Lidar (x=avant, y=gauche, z=haut)
            Tr = self.calib
            Pose_cam = self.poses[idx]
            T_world_velo = np.linalg.inv(Tr) @ Pose_cam @ Tr
            
            hom_points = np.hstack([points, np.ones((len(points), 1))])
            points = (T_world_velo @ hom_points.T).T[:, :3]
        return points

    def get_labels(self, idx):
        if idx >= len(self.label_files): return None, None
        label = np.fromfile(self.label_files[idx], dtype=np.uint32)
        return label & 0xFFFF, label >> 16

    def __len__(self): return len(self.scan_files)

In [ ]:
# @title 3.1 Construction du Nuage 4D
import numpy as np


# Mapping officiel SemanticKITTI (fusionne les classes "moving" avec leur équivalent statique)
LEARNING_MAP = {
  0: 0, 1: 0, 10: 1, 11: 2, 13: 5, 15: 3, 16: 5, 18: 4, 20: 5, 30: 6, 
  31: 7, 32: 8, 40: 9, 44: 10, 48: 11, 49: 12, 50: 13, 51: 14, 52: 0, 
  60: 9, 70: 15, 71: 16, 72: 17, 80: 18, 81: 19, 99: 0, 252: 1, 
  253: 7, 254: 6, 255: 8, 256: 5, 257: 5, 258: 4, 259: 5
}
# Vecteur de mapping rapide (taille max 260)
max_key = max(LEARNING_MAP.keys())
LABEL_MAP_ARRAY = np.zeros(max_key + 1, dtype=np.uint32)
for k, v in LEARNING_MAP.items():
    LABEL_MAP_ARRAY[k] = v

# SemanticKITTI classes "things" dans l'espace mappé (1 à 8)
# 1:car, 2:bicycle, 3:motorcycle, 4:truck, 5:other-vehicle, 6:person, 7:bicyclist, 8:motorcyclist
THINGS_CLASSES = set([1, 2, 3, 4, 5, 6, 7, 8])

# Initialisation des variables pour éviter les NameError
X_clustering = None
X_4d = None
Y_sem = None
Y_inst = None
Time_idx = None
Original_Indices = None # Pour garder la trace si on filtre

try:
    loader = SemanticKITTILoader("/content/semantic_kitti_data", SEQUENCE_TO_TEST)
    points_4d, gt_sem, gt_inst, times, indices = [], [], [], [], []
    
    total_points = 0
    actual_num_frames = len(loader) - START_FRAME if NUM_FRAMES == -1 else NUM_FRAMES
    print(f"Chargement frames {START_FRAME} -> {START_FRAME + actual_num_frames}...")
    for i in range(actual_num_frames):
        idx = START_FRAME + i
        if idx >= len(loader): break

        pts = loader.get_scan(idx, apply_pose=True)
        s_raw, inst = loader.get_labels(idx)
        
        # Mapping des labels sémantiques bruts vers l'espace d'évaluation
        s = LABEL_MAP_ARRAY[s_raw]

        # 4D Point: x, y, z, t
        t_col = np.full((len(pts), 1), i * DT_SCALE)
        pts_4d = np.hstack([pts[:, 0:3], t_col]) # Colonnes: 0:x, 1:y, 2:z, 3:t
        
        # Filtre Sémantique
        if SEMANTIC_MODE == "Oracle":
            # On ne garde que les classes "Things"
            mask = np.array([sem in THINGS_CLASSES for sem in s])
            pts_4d = pts_4d[mask]
            s = s[mask]
            inst = inst[mask]
            
            # Si on veut garder l'index original par rapport à la frame (pour de la visulaisation par ex)
            frame_indices = np.arange(len(mask))[mask]
        else:
            frame_indices = np.arange(len(pts))

        points_4d.append(pts_4d)
        gt_sem.append(s)
        gt_inst.append(inst)
        times.extend([i] * len(pts_4d))
        indices.append(frame_indices + total_points)
        total_points += len(pts) # On ajoute le total brut pour les indices absolus

    if points_4d:
        X_4d = np.vstack(points_4d)
        Y_sem = np.hstack(gt_sem)
        Y_inst = np.hstack(gt_inst)
        Time_idx = np.array(times)
        Original_Indices = np.hstack(indices)

        # Bird's Eye View : on utilise uniquement x, y, t pour le clustering
        if APPLY_BEV:
            print(f"Mode Bird's-Eye-View (BEV) activé : Clustering sur (x, y) [Streaming 2D BEV].")
            X_clustering = np.column_stack([X_4d[:, 0], X_4d[:, 1]])
        else:
            print("Mode 4D Complet activé : Clustering sur (x, y, z) [Streaming 3D].")
            X_clustering = X_4d

        print(f"Sémantique : Mode {SEMANTIC_MODE}.")
        print(f"Nuage 4D filtré: {X_4d.shape} points conservés.")
        print(f"Input Clustering: {X_clustering.shape}")
    else:
        print("Aucun point chargé ou aucun point 'thing' trouvé. Vérifiez les chemins.")

except Exception as e:
    print(f"Erreur lors du chargement des données: {e}")
    print("---------------------------------------------------------")
    print("⚠️ GÉNÉRATION DE DONNÉES SYNTHÉTIQUES (FALLBACK) ⚠️")
    print("---------------------------------------------------------")
    from sklearn.datasets import make_blobs
    n_samples = 5000
    X_syn, y_syn = make_blobs(n_samples=n_samples, n_features=3, centers=5, cluster_std=1.0)
    synth_frames = NUM_FRAMES if NUM_FRAMES != -1 else 10
    t_syn = np.random.randint(0, synth_frames, size=n_samples) * DT_SCALE
    X_4d = np.column_stack([X_syn, t_syn])
    Y_sem = np.zeros(n_samples, dtype=int)
    Y_inst = y_syn + 1 
    Time_idx = (t_syn / DT_SCALE).astype(int)
    X_clustering = X_4d[:, :3] if not APPLY_BEV else X_4d[:, [0, 1]]
    print(f"Données synthétiques générées: {X_clustering.shape}")

In [ ]:
# @title 4.1 HGP Clustering & Advanced 4D Panoptic Tracking (Coarse-to-Fine UOT GPU)
import time
import numpy as np
import os
import torch
import sys

# Ajout du chemin pour importer les modules locaux
if "/content/HGP-clusterer/tests/SemanticKITTI" not in sys.path:
    sys.path.append("/content/HGP-clusterer/tests/SemanticKITTI")
from tracker import CoarseToFineUOTTracker

# --- Import sécurisé HGP ---
try:
    from hgp_clusterer import HGPClusterer
except ImportError:
    if "/content/HGP-clusterer/src" not in sys.path:
        sys.path.append("/content/HGP-clusterer/src")
    from hgp_clusterer import HGPClusterer

# --- Paramètres ---
K = 3 # @param {type:"integer"}
MIN_CLUSTER_SIZE = 2 # @param {type:"integer"}
EXP_Z = 1 # @param {type:"number"}
SPLITTING_MODE = "uot_CrossEntropy" # @param ["None", "oracle_RandIndex", "uot_CrossEntropy", "uot_SoftRandIndex", "uot_VariationOfInformation"]
DBSCAN_FACTOR = 0.5 # @param {type:"number"}
TRACKING_VERBOSE = True # @param {type:"boolean"}
HGP_VERBOSE = False # @param {type:"boolean"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CURRENT_GT_INSTANCES = None

# ---- Règle de Split : Optimisation du Rand Index Local (ΔRI > 0) ----
def oracle_RandIndex(parent_pts_idx, children_pts_idx_list):
    global CURRENT_GT_INSTANCES
    if CURRENT_GT_INSTANCES is None or len(parent_pts_idx) == 0: return False
        
    valid_children = [c for c in children_pts_idx_list if len(c) > 0]
    if len(valid_children) <= 1:
        return False
        
    labels_parent = CURRENT_GT_INSTANCES[parent_pts_idx]
    valid_parent_mask = labels_parent != 0
    labels_parent_valid = labels_parent[valid_parent_mask]
    if len(labels_parent_valid) < 2: return False
    
    # 1. Fonction interne pour calculer le Macro-F1 (moyenne des F1-scores des instances pures)
    def compute_macro_f1(clusters_labels_list):
        unique_gts = np.unique(labels_parent_valid)
        f1_scores = []
        for gt in unique_gts:
            total_gt = np.sum(labels_parent_valid == gt)
            best_f1 = 0.0
            for cluster in clusters_labels_list:
                if len(cluster) == 0: continue
                tp = np.sum(cluster == gt)
                if tp == 0: continue
                precision = tp / len(cluster)
                recall = tp / total_gt
                f1 = 2 * (precision * recall) / (precision + recall)
                if f1 > best_f1: best_f1 = f1
            f1_scores.append(best_f1)
        return np.mean(f1_scores)
        
    # 2. F1-score du Parent (Avant le split)
    f1_parent = compute_macro_f1([labels_parent_valid])
    
    # 3. F1-score des Enfants (Après le split)
    children_labels = []
    for c_idx in valid_children:
        l_c = CURRENT_GT_INSTANCES[c_idx]
        children_labels.append(l_c[l_c != 0])
        
    f1_children = compute_macro_f1(children_labels)
    
    # 4. Décision : On n'accepte que si l'intégrité globale des instances (IoU) s'améliore !
    return f1_children > f1_parent

# ---- UOT-Guided Splitting (Cross-Entropy, Soft Rand Index, VI) ----
CURRENT_UOT_VECTORS = None # Matrice V non-normalisée globale
CURRENT_ACTIVE_TRACKS_LENGTHS = None # Tailles des tracks actives
CURRENT_POINTS_3D = None
CURRENT_PRIOR = None

def get_cluster_W(indices):
    if len(indices) == 0 or CURRENT_ACTIVE_TRACKS_LENGTHS is None: 
        return np.zeros(1) if CURRENT_ACTIVE_TRACKS_LENGTHS is None else np.zeros(len(CURRENT_ACTIVE_TRACKS_LENGTHS) + 1)
    
    S_C = np.sum(CURRENT_UOT_VECTORS[indices], axis=0) # shape (M,)
    W_C = np.zeros(len(CURRENT_ACTIVE_TRACKS_LENGTHS) + 1)
    
    for m, length in enumerate(CURRENT_ACTIVE_TRACKS_LENGTHS):
        W_C[m] = S_C[m] / max(1, length)
        
    sum_W = np.sum(W_C[:-1])
    if sum_W > 1.0:
        W_C[:-1] /= sum_W
        W_C[-1] = 0.0
    else:
        W_C[-1] = 1.0 - sum_W
    return W_C

def uot_entropy(W):
    return -np.sum(W * np.log(W + 1e-12))

def get_geometric_status(indices):
    global CURRENT_POINTS_3D, CURRENT_PRIOR
    if CURRENT_POINTS_3D is None or CURRENT_PRIOR is None or len(indices) < 3: return "UNKNOWN"
    
    pts = CURRENT_POINTS_3D[indices]
    centroid = np.mean(pts, axis=0)
    centered = pts - centroid
    
    try:
        cov = np.cov(centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
    except:
        return "UNKNOWN"
        
    # Project points on principal components
    projections = centered @ eigenvectors
    dims = np.ptp(projections, axis=0) # max - min along each axis
    
    emp_dims = np.sort(dims)[::-1] # descending
    prior_dims = np.sort([CURRENT_PRIOR.get("L", 1.0), CURRENT_PRIOR.get("W", 1.0), CURRENT_PRIOR.get("H", 1.0)])[::-1]
    
    # Check if any empirical dimension exceeds 1.2x of priors (20% margin) -> TOO BIG
    if np.any(emp_dims > prior_dims * 1.2):
        return "TOO_BIG"
    return "OK"

def uot_CrossEntropy_split(parent_indices, children_indices_list):
    global CURRENT_UOT_VECTORS, CURRENT_ACTIVE_TRACKS_LENGTHS
    if CURRENT_UOT_VECTORS is None or len(parent_indices) == 0: return False
    
    geo_status = get_geometric_status(parent_indices)
    if geo_status == "TOO_BIG":
        return True # Force split!
        
    W_p = get_cluster_W(parent_indices)
    
    # Geometric Fallback for NEW instances
    if W_p[-1] == np.max(W_p):
        if geo_status == "OK":
            return False
            
    H_p = uot_entropy(W_p)
    
    H_children = 0.0
    for child_indices in children_indices_list:
        if len(child_indices) == 0: continue
        W_c = get_cluster_W(child_indices)
        H_children += (len(child_indices) / len(parent_indices)) * uot_entropy(W_c)
        
    return H_p - H_children > 0.05

def uot_SoftRandIndex_split(parent_indices, children_indices_list):
    global CURRENT_UOT_VECTORS, CURRENT_ACTIVE_TRACKS_LENGTHS
    if CURRENT_UOT_VECTORS is None or len(parent_indices) == 0: return False
    
    geo_status = get_geometric_status(parent_indices)
    if geo_status == "TOO_BIG":
        return True # Force split!
        
    W_p = get_cluster_W(parent_indices)
    
    # Geometric Fallback for NEW instances
    if W_p[-1] == np.max(W_p):
        if geo_status == "OK":
            return False
            
    P_p = np.sum(W_p**2)
    
    P_children = 0.0
    for child_indices in children_indices_list:
        if len(child_indices) == 0: continue
        W_c = get_cluster_W(child_indices)
        P_children += (len(child_indices) / len(parent_indices)) * np.sum(W_c**2)
        
    return P_children - P_p > 0.02

def uot_VariationOfInformation_split(parent_indices, children_indices_list):
    global CURRENT_UOT_VECTORS, CURRENT_ACTIVE_TRACKS_LENGTHS
    if CURRENT_UOT_VECTORS is None or len(parent_indices) == 0: return False
    
    geo_status = get_geometric_status(parent_indices)
    if geo_status == "TOO_BIG":
        return True # Force split!
        
    W_p = get_cluster_W(parent_indices)
    
    # Geometric Fallback for NEW instances
    if W_p[-1] == np.max(W_p):
        if geo_status == "OK":
            return False
            
    H_X = uot_entropy(W_p)
    
    H_X_given_Y = 0.0
    for child_indices in children_indices_list:
        if len(child_indices) == 0: continue
        W_c = get_cluster_W(child_indices)
        H_X_given_Y += (len(child_indices) / len(parent_indices)) * uot_entropy(W_c)
        
    I_XY = H_X - H_X_given_Y
    return I_XY > 0.05

SPLITTING_REGISTRY = {
    "None": None,
    "oracle_RandIndex": oracle_RandIndex,
    "uot_CrossEntropy": uot_CrossEntropy_split,
    "uot_SoftRandIndex": uot_SoftRandIndex_split,
    "uot_VariationOfInformation": uot_VariationOfInformation_split
}

# ---- Priors Géométriques et Cinématiques ----
ALPINE_PRIORS = {
    1: {"name": "car",           "L": 4.5,  "W": 1.85, "H": 1.6,  "max_speed": 35.0, "match_threshold": 8.0, "tau": 4.0, "min_hits": 1},
    2: {"name": "bicycle",       "L": 1.8,  "W": 0.6,  "H": 1.1,  "max_speed": 15.0, "match_threshold": 3.0, "tau": 1.5, "min_hits": 3},
    3: {"name": "motorcycle",    "L": 2.2,  "W": 0.9,  "H": 1.3,  "max_speed": 35.0, "match_threshold": 3.0, "tau": 1.5, "min_hits": 3},
    4: {"name": "truck",         "L": 10.0, "W": 2.6,  "H": 3.5,  "max_speed": 30.0, "match_threshold": 30.0, "tau": 15.0, "min_hits": 1},
    5: {"name": "other-vehicle", "L": 12.0, "W": 2.6,  "H": 3.5,  "max_speed": 30.0, "match_threshold": 30.0, "tau": 15.0, "min_hits": 1},
    6: {"name": "person",        "L": 0.6,  "W": 0.6,  "H": 1.75, "max_speed": 5.0,  "match_threshold": 2.0, "tau": 1.0, "min_hits": 3},
    7: {"name": "bicyclist",     "L": 1.8,  "W": 0.75, "H": 1.8,  "max_speed": 15.0, "match_threshold": 3.0, "tau": 1.5, "min_hits": 3},
    8: {"name": "motorcyclist",  "L": 2.2,  "W": 0.9,  "H": 1.8,  "max_speed": 35.0, "match_threshold": 3.0, "tau": 1.5, "min_hits": 3},
}

def estimate_obb(pts_3d):
    """Extraction simple et passive (OOBB)"""
    if len(pts_3d) < 3:
        mins, maxs = np.min(pts_3d, axis=0), np.max(pts_3d, axis=0)
        return np.mean([mins, maxs], axis=0), np.maximum(maxs-mins, 0.1), 0.0
    pts_2d = pts_3d[:, :2]
    centroid_2d = np.mean(pts_2d, axis=0)
    pts_centered = pts_2d - centroid_2d
    try:
        u, s, vh = np.linalg.svd(pts_centered.T @ pts_centered)
        yaw = np.arctan2(u[1, 0], u[0, 0])
    except: yaw = 0.0
    cos_y, sin_y = np.cos(-yaw), np.sin(-yaw)
    R = np.array([[cos_y, -sin_y], [sin_y, cos_y]])
    pts_local = pts_centered @ R.T
    mins_l, maxs_l = np.min(pts_local, axis=0), np.max(pts_local, axis=0)
    L, W = max(0.1, maxs_l[0] - mins_l[0]), max(0.1, maxs_l[1] - mins_l[1])
    z_min, z_max = np.min(pts_3d[:, 2]), np.max(pts_3d[:, 2])
    return np.array([centroid_2d[0], centroid_2d[1], (z_max + z_min)/2]), np.array([L, W, max(0.1, z_max - z_min)]), yaw

# Initialisation
labels_pred = np.full(len(X_clustering), -1, dtype=int)
raw_labels = np.full(len(X_clustering), -1, dtype=int)
next_raw_id = 1

# Initialisation du Tracker UOT Coarse-to-Fine GPU
DT = 0.1
tracker = CoarseToFineUOTTracker(dt=DT, max_age=5, device=DEVICE, verbose=TRACKING_VERBOSE)

frames = np.unique(Time_idx)

# Boucle Streaming
for t in frames:
    f_mask = (Time_idx == t)
    global_indices = np.where(f_mask)[0]
    if TRACKING_VERBOSE: print(f"\n{'='*60}\nFRAME {t} (Indices globaux: {len(global_indices)})\n{'='*60}")
    
    # Prédit les fantômes (Extrapolation GPU)
    tracker.predict_all()
    
    classes_in_frame = np.unique(Y_sem[f_mask])
    if not TRACKING_VERBOSE: print(f"\rFrame {t} | Classes: {len(classes_in_frame)} | Active Tracks: {len(tracker.tracks)}", end="")

    for sem_cl in classes_in_frame:
        if sem_cl not in THINGS_CLASSES and SEMANTIC_MODE == "Oracle": continue
        
        cl_mask = (Y_sem[f_mask] == sem_cl)
        X_cl = X_clustering[f_mask][cl_mask]
        if len(X_cl) < MIN_CLUSTER_SIZE: continue
        
        CURRENT_GT_INSTANCES = Y_inst[f_mask][cl_mask]
        prior = ALPINE_PRIORS.get(sem_cl, {"L": 1.0, "max_speed": 20.0, "name": f"cl_{sem_cl}"})
        if TRACKING_VERBOSE: print(f"  [Process] Classe {prior.get('name')} ({sem_cl}) : {len(X_cl)} points.")
        
        spatial_X = X_cl[:, :2] if APPLY_BEV else X_cl[:, :3]
        
                # --- 1. UOT Massif Point-à-Point (Tracking-Driven) ---
        pts_gpu = torch.tensor(X_4d[global_indices[cl_mask]][:, :3], device=DEVICE, dtype=torch.float32)
        global CURRENT_UOT_VECTORS, CURRENT_ACTIVE_TRACKS_LENGTHS, CURRENT_POINTS_3D, CURRENT_PRIOR
        CURRENT_POINTS_3D = X_4d[global_indices[cl_mask]][:, :3]
        CURRENT_PRIOR = prior
        if "uot" in SPLITTING_MODE:
            V, active_tracks = tracker.compute_massive_uot(pts_gpu, sem_cl, prior)
            CURRENT_ACTIVE_TRACKS_LENGTHS = [tr.last_points_gpu.shape[0] for tr in active_tracks]
            CURRENT_UOT_VECTORS = V
        else:
            active_tracks = [tr for tr in tracker.tracks if tr.semantic_class == sem_cl]
            V = None
            CURRENT_ACTIVE_TRACKS_LENGTHS = None
            CURRENT_UOT_VECTORS = None
            
        
        # --- 2. HGP Clustering Spatial (Guidé par UOT) ---
        clusterer = HGPClusterer(
            K=K, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=K+1,
            method=prior["L"]*DBSCAN_FACTOR, expZ=EXP_Z,
            splitting=SPLITTING_REGISTRY.get(SPLITTING_MODE),
            backend=BACKEND, cgal_root=os.environ.get("CGALDELAUNAY_ROOT"), verbose=HGP_VERBOSE
        )
        try: 
            labels_cl = clusterer.fit_predict(spatial_X)
        except Exception as e:
            continue

        # --- Extraction Géométrique (OBB) & Préparation Tenseurs GPU ---
        detections = []
        u_labels = np.unique(labels_cl[labels_cl >= 0])
        if TRACKING_VERBOSE: print(f"    - HGP a trouvé {len(u_labels)} clusters spatiaux.")
        for cid in np.unique(labels_cl[labels_cl >= 0]):
            m_loc = (labels_cl == cid)
            raw_labels[global_indices[cl_mask][m_loc]] = next_raw_id
            
            # Points de l'instance pour le Fine-Matching (UOT Point-à-Point)
            pts_np = X_4d[global_indices[cl_mask][m_loc]][:, :3]
            pts_gpu = torch.tensor(pts_np, device=DEVICE, dtype=torch.float32)
            
            c, d, y = estimate_obb(pts_np)
            
            detections.append({
                "centroid": c, 
                "dimensions": d, 
                "yaw": y, 
                "points_gpu": pts_gpu, # Crucial pour le tracker
                "mask_local": m_loc
            })
            next_raw_id += 1
            
        # --- 3. Assignation Stratifiée (Actifs -> Fantômes -> Nouveaux) ---
        assigned_ids = tracker.step_assign(detections, active_tracks, V, sem_cl, prior)
        
        # --- Enregistrement des labels ---
        for m, det in enumerate(detections):
            track_id = assigned_ids[m]
            labels_pred[global_indices[cl_mask][det["mask_local"]]] = track_id

print(f"\nTracking terminé. Total IDs uniques : {tracker.next_id - 1}")



In [ ]:
# @title 5.1 Évaluation Officielle SemanticKITTI (PQ, SQ, RQ)
import os
import shutil
import numpy as np
import yaml

if X_clustering is not None and len(X_clustering) > 0:
    print("Préparation des fichiers pour l'évaluation officielle (semantic-kitti-api)...")
    
    eval_dir = "/content/eval_data"
    pred_dir = "/content/eval_predictions"
    seq_str = f"{SEQUENCE_TO_TEST:02d}"
    
    gt_labels_dir = os.path.join(eval_dir, "sequences", seq_str, "labels")
    pred_labels_dir = os.path.join(pred_dir, "sequences", seq_str, "predictions")
    
    # Nettoyage précédent éventuel
    shutil.rmtree(eval_dir, ignore_errors=True)
    shutil.rmtree(pred_dir, ignore_errors=True)
    
    os.makedirs(gt_labels_dir, exist_ok=True)
    os.makedirs(pred_labels_dir, exist_ok=True)
    
    # Création d'une configuration personnalisée pour n'évaluer que cette séquence
    custom_cfg_path = "/content/custom_eval_config.yaml"
    with open("/content/semantic-kitti-api/config/semantic-kitti.yaml", 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['split']['valid'] = [SEQUENCE_TO_TEST]
    with open(custom_cfg_path, 'w') as f:
        yaml.dump(cfg, f)
    
    # On reconstruit les labels prédits frame par frame
    actual_num_frames = len(loader) - START_FRAME if NUM_FRAMES == -1 else NUM_FRAMES
    for i in range(actual_num_frames):
        idx = START_FRAME + i
        if idx >= len(loader): break
        
        # Copie du GT
        gt_file = loader.label_files[idx]
        shutil.copy(gt_file, os.path.join(gt_labels_dir, os.path.basename(gt_file)))
        
        # Récupération des labels originaux pour garder la sémantique de fond
        s_raw, _ = loader.get_labels(idx)
        s_mapped = LABEL_MAP_ARRAY[s_raw]
        
        if SEMANTIC_MODE == "Oracle":
            mask = np.array([sem in THINGS_CLASSES for sem in s_mapped])
            frame_indices = np.where(mask)[0]
        else:
            frame_indices = np.arange(len(s_raw))
            
        # Initialisation de la prédiction
        pred_label = s_raw.astype(np.uint32)
        
        mask_time = (Time_idx == i)
        inst_preds = labels_pred[mask_time]
        
        valid_inst_mask = inst_preds >= 0
        valid_local_indices = frame_indices[valid_inst_mask]
        # Offset +1 car l'ID 0 est réservé au background
        valid_inst_ids = inst_preds[valid_inst_mask] + 1 
        
        pred_label[valid_local_indices] = (s_raw[valid_local_indices] & 0xFFFF) | (valid_inst_ids.astype(np.uint32) << 16)
        
        # Sauvegarde
        pred_filename = os.path.join(pred_labels_dir, os.path.basename(gt_file))
        pred_label.tofile(pred_filename)
        
    print("Fichiers de prédiction générés. Téléchargement des scripts d'évaluation 4D...")
    os.system("wget -q https://raw.githubusercontent.com/MehmetAygun/4D-PLS/master/utils/evaluate_4dpanoptic.py -O /content/semantic-kitti-api/evaluate_4dpanoptic.py")
    os.system("wget -q https://raw.githubusercontent.com/MehmetAygun/4D-PLS/master/utils/eval_np.py -O /content/semantic-kitti-api/auxiliary/eval_np_4d.py")
    os.system("sed -i 's/from eval_np import Panoptic4DEval/from auxiliary.eval_np_4d import Panoptic4DEval/' /content/semantic-kitti-api/evaluate_4dpanoptic.py")
    
    os.makedirs("/content/eval_output", exist_ok=True)
    # On lance le script officiel 4D sur ce mini-dataset de test et on capture la sortie
    import subprocess
    print("Évaluation en cours (cela peut prendre quelques minutes)...")
    eval_cmd = f"python3 /content/semantic-kitti-api/evaluate_4dpanoptic.py --dataset {eval_dir} --predictions {pred_dir} --split valid --data_cfg {custom_cfg_path}"
    res = subprocess.run(eval_cmd, shell=True, capture_output=True, text=True)
    
    print("\n--- RÉSULTATS DE L'ÉVALUATION LSTQ (4D) ---")
    print(res.stdout)
    if res.stderr:
        print("Erreurs :", res.stderr)
    
    print("\n--- RÉSULTATS NORMALISÉS 4D (Classes présentes uniquement) ---")
    try:
        import re
        # Extraction des tableaux Numpy de la sortie brute
        aq_match = re.search(r'Assoc:\s*\[(.*?)\]', res.stdout, re.DOTALL)
        iou_match = re.search(r'iou:\s*\[(.*?)\]', res.stdout, re.DOTALL)
        
        if aq_match and iou_match:
            aq_arr = [float(x) for x in aq_match.group(1).replace('\n', '').split()]
            iou_arr = [float(x) for x in iou_match.group(1).replace('\n', '').split()]
            
            present_sem_classes = np.unique(Y_sem)
            things_names_mapping = {1: "car", 2: "bicycle", 3: "motorcycle", 4: "truck", 5: "other-vehicle", 6: "person", 7: "bicyclist", 8: "motorcyclist"}
            present_things_ids = [c for c in present_sem_classes if c in things_names_mapping]
            
            if present_things_ids:
                aq_sum, iou_sum, lstq_sum = 0.0, 0.0, 0.0
                for cid in present_things_ids:
                    c_aq = aq_arr[cid]
                    c_iou = iou_arr[cid]
                    c_lstq = np.sqrt(c_aq * c_iou)
                    
                    aq_sum += c_aq
                    iou_sum += c_iou
                    lstq_sum += c_lstq
                    
                n = len(present_things_ids)
                print(f"Classes Things présentes ({n}) : {', '.join([things_names_mapping[c] for c in present_things_ids])}")
                print(f"LSTQ_things_normalized: {lstq_sum / n:.4f}")
                print(f"S_cls (IoU)_things_normalized: {iou_sum / n:.4f}")
                print(f"S_assoc (AQ)_things_normalized: {aq_sum / n:.4f}")
        else:
            print("Impossible de parser la sortie du script.")
    except Exception as e:
        print(f"Erreur lors du calcul normalisé : {e}")
else:
    print("Pas de données pour l'évaluation.")


In [ ]:
# @title 6.1 Visualisation 4D (Monde LiDAR : X,Y = Sol | Z = Temps)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

VIZ_START_FRAME = 0 # @param {type:"integer"}
VIZ_END_FRAME = 200 # @param {type:"integer"}

if X_4d is not None and len(X_4d) > 0:
    # Filtrage par frames pour l'affichage
    frame_mask = (Time_idx >= VIZ_START_FRAME) & (Time_idx <= VIZ_END_FRAME)
    X_4d_viz = X_4d[frame_mask]
    labels_pred_viz = labels_pred[frame_mask]
    Y_inst_viz = Y_inst[frame_mask]
    Y_sem_viz = Y_sem[frame_mask]
    raw_labels_viz = raw_labels[frame_mask]
    Time_idx_viz = Time_idx[frame_mask]
    
    max_points = 200000
    if len(X_4d_viz) > max_points:
        idx = np.random.choice(len(X_4d_viz), max_points, replace=False)
    else:
        idx = np.arange(len(X_4d_viz))
        
    X_v, L_v, GT_v, S_v, R_v, T_v = X_4d_viz[idx].copy(), labels_pred_viz[idx], Y_inst_viz[idx], Y_sem_viz[idx], raw_labels_viz[idx], Time_idx_viz[idx]
    
    THINGS_NAMES = {1: "car", 2: "bicycle", 3: "motorcycle", 4: "truck", 5: "other-vehicle", 6: "person", 7: "bicyclist", 8: "motorcyclist"}
    
    fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'scene'}, {'type': 'scene'}, {'type': 'scene'}]],
                        subplot_titles=('Ground Truth', 'Raw Clustering', 'Panoptic Tracking'),
                        horizontal_spacing=0.02)
    
    def add_traces(ids, col, name_prefix):
        m = ids > -1
        if not np.any(m): return
        m_x, m_y, m_z, m_ids, m_s, m_gt, m_t = X_v[m, 0], X_v[m, 1], X_v[m, 3], ids[m], S_v[m], GT_v[m], T_v[m]
        
        if name_prefix == "Track":
            hover_texts = [f"Frame: {t}<br>Class: {THINGS_NAMES.get(s, 'unknown')}<br>Track ID: {i}<br>GT ID: {gt}" for t, s, i, gt in zip(m_t, m_s, m_ids, m_gt)]
        else:
            hover_texts = [f"Frame: {t}<br>Class: {THINGS_NAMES.get(s, 'unknown')}<br>ID: {i}" for t, s, i in zip(m_t, m_s, m_ids)]
        
        # Hachage simple pour disperser les couleurs (surtout utile pour la vérité terrain)
        color_values = (m_ids * 97) % 256
        
        fig.add_trace(go.Scatter3d(
            x=m_x, y=m_y, z=m_z,
            mode='markers', 
            marker=dict(size=2, color=color_values, colorscale='Turbo'), 
            name=name_prefix,
            text=hover_texts, hoverinfo='text+name'
        ), row=1, col=col)

    add_traces(GT_v, 1, "GT")
    add_traces(R_v, 2, "Raw")
    add_traces(L_v, 3, "Track")
    
    scene_config = dict(
        aspectmode='data', # Orthonormalité garantie
        xaxis_title='X Monde (m)', yaxis_title='Y Monde (m)', zaxis_title='Temps (s)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    )
    
    fig.update_layout(title_text="Segmentation Panoptique 4D (Référentiel Monde LiDAR)", 
                      height=850, showlegend=False,
                      scene=scene_config, scene2=scene_config, scene3=scene_config)
    fig.show()
else:
    print("Pas de données à afficher.")



In [ ]:
# @title 6.1.bis Visualisation 3D d'une Frame Unique (Qualité Papier)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FRAME_TO_SHOW = 0 # @param {type:"integer"}

# SemanticKITTI colors (BGR to RGB)
SEMANTIC_COLORS = {
  0 : [0, 0, 0],
  1 : [255, 0, 0],
  10: [100, 150, 245],
  11: [100, 230, 245],
  13: [100, 80, 250],
  15: [30, 60, 150],
  16: [0, 0, 255],
  18: [80, 30, 180],
  20: [0, 0, 255],
  30: [255, 30, 30],
  31: [255, 40, 200],
  32: [150, 30, 90],
  40: [255, 0, 255],
  44: [255, 150, 255],
  48: [75, 0, 75],
  49: [175, 0, 75],
  50: [255, 200, 0],
  51: [255, 120, 50],
  52: [255, 150, 0],
  60: [150, 255, 170],
  70: [0, 175, 0],
  71: [135, 60, 0],
  72: [150, 240, 80],
  80: [255, 240, 150],
  81: [255, 0, 255],
  99: [50, 255, 255],
  252: [100, 150, 245],
  256: [0, 0, 255],
  253: [255, 40, 200],
  254: [255, 30, 30],
  255: [150, 30, 90],
  257: [100, 80, 250],
  258: [80, 30, 180],
  259: [0, 0, 255]
}

SEMANTIC_NAMES = {
  0 : "unlabeled",
  1 : "outlier",
  10: "car",
  11: "bicycle",
  13: "bus",
  15: "motorcycle",
  16: "on-rails",
  18: "truck",
  20: "other-vehicle",
  30: "person",
  31: "bicyclist",
  32: "motorcyclist",
  40: "road",
  44: "parking",
  48: "sidewalk",
  49: "other-ground",
  50: "building",
  51: "fence",
  52: "other-structure",
  60: "lane-marking",
  70: "vegetation",
  71: "trunk",
  72: "terrain",
  80: "pole",
  81: "traffic-sign",
  99: "other-object",
  252: "moving-car",
  256: "moving-on-rails",
  253: "moving-bicyclist",
  254: "moving-person",
  255: "moving-motorcyclist",
  257: "moving-bus",
  258: "moving-truck",
  259: "moving-other-vehicle"
}

STUFF_CLASSES = [40, 44, 48, 49, 50, 51, 52, 60, 70, 71, 72, 80, 81, 99]

if X_4d is not None and len(X_4d) > 0:
    frame_mask = (Time_idx == FRAME_TO_SHOW)
    
    if np.any(frame_mask):
        X_frame = X_4d[frame_mask]
        labels_frame = labels_pred[frame_mask]
        Y_sem_frame = Y_sem[frame_mask]
        GT_frame = Y_inst[frame_mask]
        
        # Sous-échantillonnage optionnel pour affichage fluide
        max_points_frame = 100000
        if len(X_frame) > max_points_frame:
            idx = np.random.choice(len(X_frame), max_points_frame, replace=False)
            X_frame = X_frame[idx]
            labels_frame = labels_frame[idx]
            Y_sem_frame = Y_sem_frame[idx]
            GT_frame = GT_frame[idx]
            
        fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                            subplot_titles=('Ground Truth', 'Clustering & Tracking'),
                            horizontal_spacing=0.02)
        
        def add_frame_traces(col, ids, is_gt=False):
            # STUFF points
            stuff_mask = np.isin(Y_sem_frame, STUFF_CLASSES)
            if np.any(stuff_mask):
                # Convert to rgba with 0.15 opacity for stuff
                stuff_colors = np.array([SEMANTIC_COLORS.get(c, [128, 128, 128]) for c in Y_sem_frame[stuff_mask]])
                stuff_colors_str = [f'rgba({r},{g},{b},0.15)' for r, g, b in stuff_colors]
                hover_texts = [f"Class: {SEMANTIC_NAMES.get(c, 'unknown')} ({c})" for c in Y_sem_frame[stuff_mask]]
                
                fig.add_trace(go.Scatter3d(
                    x=X_frame[stuff_mask, 0], y=X_frame[stuff_mask, 1], z=X_frame[stuff_mask, 2],
                    mode='markers',
                    marker=dict(size=1.5, color=stuff_colors_str),
                    name='STUFF',
                    text=hover_texts,
                    hoverinfo='text'
                ), row=1, col=col)
                
            # THINGS points
            things_mask = ~stuff_mask & (ids > -1)
            if np.any(things_mask):
                unique_ids = np.unique(ids[things_mask])
                for uid in unique_ids:
                    uid_mask = things_mask & (ids == uid)
                    if not np.any(uid_mask): continue
                    
                    # Récupération de la classe sémantique majoritaire
                    sem_class = np.bincount(Y_sem_frame[uid_mask]).argmax()
                    class_name = SEMANTIC_NAMES.get(sem_class, 'unknown')
                    base_color = SEMANTIC_COLORS.get(sem_class, [255, 255, 255])
                    
                    if is_gt:
                        color_str = f'rgb({base_color[0]},{base_color[1]},{base_color[2]})'
                    else:
                        # Légère variation de couleur pour distinguer les instances prédites de la même classe
                        np.random.seed(uid)
                        variation = np.random.randint(-30, 30, size=3)
                        c = np.clip(np.array(base_color) + variation, 0, 255)
                        color_str = f'rgb({c[0]},{c[1]},{c[2]})'
                        
                    fig.add_trace(go.Scatter3d(
                        x=X_frame[uid_mask, 0], y=X_frame[uid_mask, 1], z=X_frame[uid_mask, 2],
                        mode='markers',
                        marker=dict(size=3, color=color_str, opacity=1.0, line=dict(width=0)),
                        name=f'{class_name} {uid}',
                        text=[f"ID: {uid}<br>Class: {class_name} ({sem_class})"] * np.sum(uid_mask),
                        hoverinfo='text'
                    ), row=1, col=col)
                    
        add_frame_traces(1, GT_frame, is_gt=True)
        add_frame_traces(2, labels_frame, is_gt=False)
        
        scene_config = dict(
            aspectmode='data',
            xaxis_title='X Monde (m)', yaxis_title='Y Monde (m)', zaxis_title='Z Monde (m)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
            bgcolor='rgb(20, 24, 34)', # Fond sombre élégant
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='')
        )
        
        fig.update_layout(title_text=f"<b>Frame {FRAME_TO_SHOW}</b> - Segmentation Panoptique 3D", 
                          title_font=dict(color='white', size=20),
                          paper_bgcolor='rgb(20, 24, 34)',
                          plot_bgcolor='rgb(20, 24, 34)',
                          height=800, showlegend=False,
                          scene=scene_config, scene2=scene_config,
                          margin=dict(l=0, r=0, b=0, t=50))
        
        # Subplot titles formatting
        for annotation in fig['layout']['annotations']:
            annotation['font'] = dict(size=16, color='white')
            
        fig.show()
    else:
        print(f"La frame {FRAME_TO_SHOW} n'est pas présente dans le sous-ensemble actuel.")
else:
    print("Pas de données à afficher.")


In [ ]:
# @title 6.2 [DEBUG] Visualisation des Scores d'Association (Matrice UOT)
import plotly.express as px
import pandas as pd

if TRACKING_VERBOSE and 'C_final' in locals():
    # On affiche la matrice de la dernière classe traitée dans la dernière frame
    fig_m = px.imshow(C_final, 
                    labels=dict(x="Détections", y="Tracks Existantes", color="Coût UOT"),
                    title="Matrice de Coût Fine-Matching (UOT Point-à-Point)",
                    color_continuous_scale='Viridis_r')
    fig_m.show()
    print("Note: Les valeurs proches de 0 indiquent une forte ressemblance géométrique.")
else:
    print("Activez TRACKING_VERBOSE pour capturer les matrices de coût.")


In [ ]:
# @title 6.3 [DEBUG] Profiling des Vitesses Estimées (Kalman)
import plotly.graph_objects as go

if tracker and len(tracker.tracks) > 0:
    fig_v = go.Figure()
    for tr in tracker.tracks:
        # Calcul de la norme de la vitesse
        v_norm = np.linalg.norm(tr.x[3:6])
        print(f"Track {tr.track_id} ({ALPINE_PRIORS.get(tr.semantic_class, {}).get('name', 'obj')}): Vitesse = {v_norm:.2f} m/s")
else:
    print("Aucune piste active à analyser.")
